```sql
Goal:
- load the joined train / validation split
- build a simple baseline feature set
- train one baseline model
- evaluate on the validation split

```

In [1]:
import pandas as pd
import numpy as np

In [2]:
from lightgbm import LGBMClassifier

In [3]:
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split

#### 1. Load Joined train validation split

In [4]:
train_df=pd.read_pickle('../data/interim/train_joined_split.pkl')

In [5]:
valid_df=pd.read_pickle('../data/interim/valid_joined_split.pkl')

In [6]:
train_df.shape, valid_df.shape

((472432, 434), (118108, 434))

In [7]:
print(f"Train fraud rate: {train_df['isFraud'].mean():.2%}")
print(f"Valid fraud rate: {valid_df['isFraud'].mean():.2%}")

Train fraud rate: 3.51%
Valid fraud rate: 3.44%


#### 2. Building Baseline Feature Set 

In [8]:
train_df.head(5)

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M


In [9]:
base_features=["TransactionAmt",
               "ProductCD",
               "card1", "card2", "card3", "card4", "card5", "card6",
               "addr1", "addr2",
               "P_emaildomain", "R_emaildomain",
               "DeviceType", "DeviceInfo"
              ]
target_col="isFraud"

In [10]:
available_features = [c for c in base_features if c in train_df.columns]

In [11]:
# Adding some transformation on features
for df_ in [train_df,valid_df]:
    df_["log_txnamt"]=np.log1p(df_["TransactionAmt"]) #log1p works best for skewed and zero values well
    df_["has_identity"]=(df_[["DeviceType", "DeviceInfo"]].notna().any(axis=1).astype(int)) #Any one value is available

In [12]:
new_features = ["log_txnamt", "has_identity"]

In [13]:
feature_cols=available_features+new_features

In [14]:
X_train=train_df[feature_cols]
X_valid=valid_df[feature_cols]

y_train=train_df[target_col]
y_valid=valid_df[target_col]

In [15]:
#Seperating Categorical and Numerical Columns
cat_cols=X_train.select_dtypes(include=["object", "string", "category"]).columns.tolist()
num_cols=X_train.select_dtypes(include="number").columns.tolist()
remaining_cols = X_train.columns.difference(cat_cols+num_cols)
print(remaining_cols)

Index([], dtype='str')


In [16]:
#Null value replacement
for col in num_cols:
    X_train[col]=X_train[col].fillna(X_train[col].median())
    X_valid[col]=X_valid[col].fillna(X_valid[col].median())
for col in cat_cols:
    X_train[col]=X_train[col].fillna("Missing").astype(str)
    X_valid[col]=X_valid[col].fillna("Missing").astype(str)

In [17]:
# Typecasting all strings into category
for col in cat_cols:
    X_train[col]=X_train[col].astype("category")
    X_valid[col]=X_valid[col].astype("category")

#### 3. Training a LighGBM Model

In [18]:
model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    class_weight="balanced"
)

In [19]:
model.fit(
    X_train,
    y_train,
    categorical_feature=cat_cols
)

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Number of positive: 16599, number of negative: 455833
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.025948 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1808
[LightGBM] [Info] Number of data points in the train set: 472432, number of used features: 16
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


,boosting_type,'gbdt'
,num_leaves,64
,max_depth,-1
,learning_rate,0.05
,n_estimators,300
,subsample_for_bin,200000
,objective,None
,class_weight,'balanced'
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [20]:
valid_pred = model.predict_proba(X_valid)[:, 1]

In [21]:
print("Prediction summary:")
print(pd.Series(valid_pred).describe())

Prediction summary:
count    118108.000000
mean          0.266851
std           0.204329
min           0.000439
25%           0.108561
50%           0.210702
75%           0.378237
max           0.996235
dtype: float64


In [22]:
roc_auc = roc_auc_score(y_valid, valid_pred)
pr_auc = average_precision_score(y_valid, valid_pred)

print(f"ROC-AUC : {roc_auc:.5f}")
print(f"PR-AUC  : {pr_auc:.5f}")

ROC-AUC : 0.80776
PR-AUC  : 0.23062


In [23]:
score_df = pd.DataFrame({
    "score": valid_pred,
    "isFraud": y_valid.values
})

score_df["score_band"] = pd.qcut(score_df["score"], q=10, duplicates="drop")

band_summary = (
    score_df.groupby("score_band")
    .agg(
        rows=("isFraud", "size"),
        fraud_rate=("isFraud", "mean"),
        fraud_count=("isFraud", "sum"),
        avg_score=("score", "mean")
    )
    .reset_index()
)

display(band_summary)

,score_band,rows,fraud_rate,fraud_count,avg_score
0,"(-0.0005610000000000001, 0.0584]",11812,0.006688,79,0.035633
1,"(0.0584, 0.0918]",11810,0.007113,84,0.075512
2,"(0.0918, 0.126]",11811,0.008213,97,0.108724
3,"(0.126, 0.165]",11810,0.011854,140,0.144739
4,"(0.165, 0.211]",11814,0.013966,165,0.187309
5,"(0.211, 0.267]",11808,0.018293,216,0.237956
6,"(0.267, 0.337]",11810,0.020068,237,0.300726
7,"(0.337, 0.426]",11816,0.031229,369,0.379451
8,"(0.426, 0.554]",11806,0.044384,524,0.482043
9,"(0.554, 0.996]",11811,0.182288,2153,0.716466


In [24]:
importance_df = pd.DataFrame({
    "feature": X_train.columns,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

display(importance_df.head(20))

,feature,importance
2,card1,3088
0,TransactionAmt,3008
8,addr1,2611
3,card2,2424
10,P_emaildomain,2156
13,DeviceInfo,1463
6,card5,917
11,R_emaildomain,774
14,log_txnamt,662
4,card3,465


```sql
## Outcome

At this point we have:
- one trusted baseline model
- validation predictions
- baseline ROC-AUC / PR-AUC
- score-band view
- first feature importance check

Next step:
- add the first real feature iteration on top of this baseline
```